# MCDD experiments

This notebook reproduces the experimental workflow used in the MCDD study on
the HDF5 streams stored under `data/datasets/`.

Included methods:

- MCDD with sliding and growing windows;
- Traditional Single Hypothesis (TSH) baselines;
- River KSWIN;
- LORD under local dependence.

SEED is intentionally not included. The former *Comparison: Sliding vs Growing
Window* section is also omitted.


## Repository setup and imports


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    """Locate the repository root from the current working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "src" / "mcdd").is_dir():
            return candidate

    raise FileNotFoundError(
        "Repository root not found. Open this notebook from inside the repository."
    )


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"

if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from mcdd.experiments import (
    PAPER_CONFIGURATIONS,
    article_table_for_drift,
    article_table_to_latex,
    average_metrics_by_drift,
    average_metrics_overall,
    configuration_table,
    evaluate_single_stream,
    expected_dataset_paths,
    format_article_table,
    get_configuration,
    overall_article_table,
    read_stream,
    run_archive_experiment,
    run_experiment_suite,
    summarize_archive_results,
    summarize_results,
)

DATA_DIRECTORY = REPOSITORY_ROOT / "data" / "datasets"
RESULTS_DIRECTORY = REPOSITORY_ROOT / "results"
PER_RUN_RESULTS = RESULTS_DIRECTORY / "per_run_results.csv"
SUMMARY_RESULTS = RESULTS_DIRECTORY / "summary_results.csv"

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Dataset directory: {DATA_DIRECTORY}")
print(f"Results directory: {RESULTS_DIRECTORY}")


## Experimental configurations


In [ ]:
configurations = configuration_table(PAPER_CONFIGURATIONS)
display(configurations)


### Scoring convention

The evaluation preserves the convention used in the original experiments:

- a valid detection must satisfy `drift_start < detection < valid_detection_end`;
- abrupt drift uses a 2,000-sample valid interval;
- gradual and incremental drift use the generated transition interval;
- a first alarm outside the valid interval is counted as a false alarm only;
- if no alarm occurs, the drift is counted as missed.


## Validate the HDF5 dataset files


In [ ]:
dataset_paths = expected_dataset_paths(DATA_DIRECTORY)

for dataset_path in dataset_paths:
    print(dataset_path.name)


## Quick validation

Run the ten detector configurations on the first stream of
`abrupt_normal.h5`. This is a functional check only; it is not part of the
reported 1,000-replication results.


In [ ]:
quick_archive = DATA_DIRECTORY / "abrupt_normal.h5"
values, metadata = read_stream(quick_archive, row_index=0)

quick_results = [
    evaluate_single_stream(
        values,
        **metadata,
        configuration=configuration,
    )
    for configuration in PAPER_CONFIGURATIONS
]

quick_results_table = pd.DataFrame(quick_results)
display(
    quick_results_table[
        [
            "configuration",
            "alarm_index",
            "outcome",
            "drift_start",
            "valid_detection_end",
            "delay",
            "TP",
            "FP",
            "FN",
        ]
    ]
)


## Run one detector on one dataset archive

This section evaluates one selected detector configuration on one selected HDF5
archive. Keep `SELECTED_MAX_STREAMS = None` to evaluate all 1,000 streams.


In [ ]:
RUN_SELECTED_EXPERIMENT = False

SELECTED_DATASET = "abrupt_normal.h5"
SELECTED_CONFIGURATION = "MCDD-S"

# None evaluates all streams in the selected archive.
SELECTED_MAX_STREAMS = None
OVERWRITE_SELECTED_RESULTS = False

if RUN_SELECTED_EXPERIMENT:
    selected_archive = DATA_DIRECTORY / SELECTED_DATASET
    selected_configuration = get_configuration(SELECTED_CONFIGURATION)

    selected_results_directory = RESULTS_DIRECTORY / "selected"
    safe_configuration_name = (
        SELECTED_CONFIGURATION.lower().replace("-", "_")
    )
    result_stem = f"{selected_archive.stem}__{safe_configuration_name}"

    selected_per_run_file = (
        selected_results_directory / f"{result_stem}_per_run.csv"
    )
    selected_summary_file = (
        selected_results_directory / f"{result_stem}_summary.csv"
    )

    run_archive_experiment(
        archive_path=selected_archive,
        configuration=selected_configuration,
        output_file=selected_per_run_file,
        max_streams=SELECTED_MAX_STREAMS,
        overwrite=OVERWRITE_SELECTED_RESULTS,
        progress_every=25,
    )

    selected_summary = summarize_archive_results(
        per_run_file=selected_per_run_file,
        output_file=selected_summary_file,
    )

    display(selected_summary)
    print(f"Per-run results: {selected_per_run_file}")
    print(f"Summary results: {selected_summary_file}")
else:
    print(
        "Selected experiment is disabled. Set "
        "RUN_SELECTED_EXPERIMENT = True when ready."
    )


## Full experiment execution

The complete paper experiment evaluates 9 HDF5 archives × 1,000 streams ×
10 detector configurations = 90,000 detector-stream runs.

For the complete experiment, keep `MAX_STREAMS_PER_ARCHIVE = None`.


In [ ]:
RUN_FULL_EXPERIMENTS = False
OVERWRITE_RESULTS = False

# None runs all 1,000 streams from every archive.
# Use a small integer such as 2 for a reduced execution test.
MAX_STREAMS_PER_ARCHIVE = None

if RUN_FULL_EXPERIMENTS:
    run_experiment_suite(
        data_directory=DATA_DIRECTORY,
        output_file=PER_RUN_RESULTS,
        configurations=PAPER_CONFIGURATIONS,
        max_streams_per_archive=MAX_STREAMS_PER_ARCHIVE,
        overwrite=OVERWRITE_RESULTS,
        progress_every=25,
    )

    summary = summarize_results(
        per_run_file=PER_RUN_RESULTS,
        output_file=SUMMARY_RESULTS,
    )
    display(summary)
else:
    print(
        "Full execution is disabled. Set RUN_FULL_EXPERIMENTS = True "
        "and run this cell again when ready."
    )


## Inspect previously generated results


In [ ]:
if PER_RUN_RESULTS.is_file():
    per_run_results = pd.read_csv(PER_RUN_RESULTS)
    print(f"Per-run rows: {len(per_run_results):,}")
    display(per_run_results.head())

    summary = summarize_results(
        per_run_file=PER_RUN_RESULTS,
        output_file=SUMMARY_RESULTS,
    )
    display(summary)
else:
    print(f"No per-run result file found at {PER_RUN_RESULTS}.")


## Article-style result tables

The paper-level tables are produced in two steps:

1. For each detector and drift type, the FDR, MDR, IR, and Mean Delay are
   averaged arithmetically across the `normal`, `exponential`, and `gamma`
   distribution rows.
2. The overall table averages the three resulting drift-level values for each
   detector.

The `distribution="all"` pooled rows in `summary_results.csv` are deliberately
not used for these article tables.

For scenarios in which a detector produces only false alarms and no valid drift
detections, the article-style `NA` convention is applied to the metrics that are
not meaningful.


In [ ]:
if SUMMARY_RESULTS.is_file():
    by_drift = average_metrics_by_drift(
        SUMMARY_RESULTS,
        article_na=True,
    )
    overall = average_metrics_overall(by_drift)

    for drift_type, title in (
        ("abrupt", "Abrupt drift"),
        ("gradual", "Gradual drift"),
        ("incremental", "Incremental drift"),
    ):
        print(f"\n{title}")
        table = article_table_for_drift(by_drift, drift_type)
        display(format_article_table(table, decimals=4))

    print("\nOverall comparison")
    overall_table = overall_article_table(overall)
    display(format_article_table(overall_table, decimals=4))
else:
    print(
        f"No summary file found at {SUMMARY_RESULTS}. "
        "Run the experiments first."
    )


### Optional LaTeX output

Set `PRINT_LATEX_TABLES = True` to print tables that can be copied into the
article source.


In [ ]:
PRINT_LATEX_TABLES = False

if PRINT_LATEX_TABLES and SUMMARY_RESULTS.is_file():
    by_drift = average_metrics_by_drift(SUMMARY_RESULTS, article_na=True)
    overall = average_metrics_overall(by_drift)

    latex_tables = {
        "abrupt": article_table_for_drift(by_drift, "abrupt"),
        "gradual": article_table_for_drift(by_drift, "gradual"),
        "incremental": article_table_for_drift(by_drift, "incremental"),
        "overall": overall_article_table(overall),
    }

    for key, table in latex_tables.items():
        print(f"\n% {key}")
        print(
            article_table_to_latex(
                table,
                decimals=4,
                label=f"tab:{key}_results",
            )
        )
